In [ ]:
# %load test4.py


# ===== 必须放在最前面 =====
import sys
import os

project_root = os.path.abspath("..")  # Debug 的上一级

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root:", project_root)
"""
示例：演示如何收集策略产生的买卖点特征，生成训练样本，训练 XGBoost 模型，
并可视化检查标签（仅作演示，训练与预测使用同一数据集仅为示例用途）。
"""

import json
from typing import Dict, TypedDict

import xgboost as xgb

from Chan import CChan
from ChanConfig import CChanConfig
from ChanModel.Features import CFeatures
from Common.CEnum import AUTYPE, DATA_SRC, KL_TYPE
from Common.CTime import CTime
from Plot import get_plot_driver
from ChanModel.feature_center import build_features

class T_SAMPLE_INFO(TypedDict):
    feature: CFeatures
    is_buy: bool
    open_time: CTime


def plot(chan, plot_marker):
    plot_config = {
        "plot_kline": True,
        "plot_bi": True,
        "plot_seg": True,
        "plot_zs": True,
        "plot_bsp": True,
        "plot_marker": True,
    }
    plot_para = {
        "figure": {
            "x_range": 400,
        },
        "marker": {
            "markers": plot_marker
        }
    }
    CPlotDriver = get_plot_driver("plotly")
    plot_driver = CPlotDriver(
        chan,
        plot_config=plot_config,
        plot_para=plot_para,
    )
    plot_driver.save2img("label.html")


if __name__ == "__main__":
    """
    本demo主要演示如何记录策略产出的买卖点的特征
    然后将这些特征作为样本，训练一个模型(以XGB为demo)
    用于预测买卖点的准确性

    请注意，demo训练预测都用的是同一份数据，这是不合理的，仅仅是为了演示
    """
    code = "BTCUSDT"  # 标的代码
    begin_time = "2020-02-04"  # 起始时间（用于回测）
    end_time = "2025-01-01"  # 结束时间，None 表示直到最新
    data_src = DATA_SRC.CSV  # 数据来源类型
    lv_list = [KL_TYPE.K_15M]

    config = CChanConfig({
        "trigger_step": True,  # 打开开关！
        "bi_strict": True,
        "skip_step": 0,
        "divergence_rate": float("inf"),
        "bsp2_follow_1": False,
        "bsp3_follow_1": False,
        "min_zs_cnt": 0,
        "bs1_peak": False,
        "macd_algo": "peak",
        "bs_type": '1,2,3a,1p,2s,3b',
        "print_warning": True,
        "zs_algo": "normal",
    })

    chan = CChan(
        code=code,
        begin_time=begin_time,
        end_time=end_time,
        data_src=data_src,
        lv_list=lv_list,
        config=config,
        autype=AUTYPE.QFQ,
    )

    bsp_dict: Dict[int, T_SAMPLE_INFO] = {}  # 存储策略产出的 bsp 的特征

    # 跑策略，保存买卖点的特征
    # 遍历按步快照，收集每个买卖点的特征（示例中以分形完成时记录）
    for chan_snapshot in chan.step_load():
        last_klu = chan_snapshot[0][-1][-1]
        bsp_list = chan_snapshot.get_latest_bsp()
        if not bsp_list:
            continue
        last_bsp = bsp_list[0]

        cur_lv_chan = chan_snapshot[0]
        # 只在买卖点第一次出现且索引对齐时记录样本特征
        if last_bsp.klu.idx not in bsp_dict and cur_lv_chan[-2].idx == last_bsp.klu.klc.idx:
            # 假如策略是：买卖点分形第三元素出现时交易
            bsp_dict[last_bsp.klu.idx] = {
                "feature": last_bsp.features,
                "is_buy": last_bsp.is_buy,
                "open_time": last_klu.time,
            }
            # 为该买卖点加入当前开仓 K 线的补充特征
            # bsp_dict[last_bsp.klu.idx]["feature"].add_feat(stragety_feature(last_klu))
            extra_feat = build_features(
                klu=last_klu,
                history=cur_lv_chan.lst,
                chan=cur_lv_chan
            )

            bsp_dict[last_bsp.klu.idx]["feature"].add_feat(extra_feat)
            print(last_bsp.klu.time, last_bsp.is_buy)

    # 将收集到的特征输出为 libsvm 格式，供 XGBoost 训练使用
    bsp_academy = [bsp.klu.idx for bsp in chan.get_latest_bsp(number=0)]
    feature_meta = {}  # 特征meta
    cur_feature_idx = 0
    plot_marker = {}
    fid = open("feature.libsvm", "w")
    for bsp_klu_idx, feature_info in bsp_dict.items():
        label = int(bsp_klu_idx in bsp_academy)  # 以买卖点识别是否准确为label
        features = []  # List[(idx, value)]
        for feature_name, value in feature_info['feature'].items():
            if feature_name not in feature_meta:
                feature_meta[feature_name] = cur_feature_idx
                cur_feature_idx += 1
            features.append((feature_meta[feature_name], value))
        features.sort(key=lambda x: x[0])
        feature_str = " ".join([f"{idx}:{value}" for idx, value in features])
        fid.write(f"{label} {feature_str}\n")
        plot_marker[feature_info["open_time"].to_str()] = ("√" if label else "×", "down" if feature_info["is_buy"] else "up")
    fid.close()

    # 保存特征 meta（实盘载入模型时需要用此 meta 对齐特征）
    with open("feature.meta", "w") as fid:
        fid.write(json.dumps(feature_meta))

    # 可视化：将带标签的买卖点画到图上便于检查
    plot(chan, plot_marker)

    # 使用 libsvm 文件训练 XGBoost 模型（示例参数）
    dtrain = xgb.DMatrix("feature.libsvm?format=libsvm")
    param = {'max_depth': 2, 'eta': 0.3, 'objective': 'binary:logistic', 'eval_metric': 'auc'}
    evals_result = {}
    bst = xgb.train(
        param,
        dtrain=dtrain,
        num_boost_round=10,
        evals=[(dtrain, "train")],
        evals_result=evals_result,
        verbose_eval=True,
    )
    bst.save_model("model.json")

    # 演示加载模型并对训练数据做一次预测（仅作完整流程展示）
    model = xgb.Booster()
    model.load_model("model.json")
    print(model.predict(dtrain))

Project root: D:\WorkSpace\python\chan.py
2020/02/04 10:45 True
2020/02/04 12:30 True
2020/02/04 13:15 True
2020/02/04 19:00 True
2020/02/04 20:00 True
2020/02/05 01:00 False
2020/02/05 02:15 True
2020/02/05 04:00 False
2020/02/05 05:00 False
2020/02/05 06:15 False
2020/02/05 07:15 False
2020/02/05 08:30 True
2020/02/05 12:15 False
2020/02/05 16:15 False
2020/02/05 18:15 False
2020/02/06 00:15 True
2020/02/06 04:15 False
2020/02/06 05:15 False
2020/02/06 06:45 False
2020/02/06 10:45 False
2020/02/06 13:15 False
2020/02/07 11:30 False
2020/02/07 22:45 False
2020/02/07 23:45 False
2020/02/08 03:00 True
2020/02/08 04:45 False
2020/02/08 06:00 False
2020/02/08 08:45 False
2020/02/08 10:15 False
2020/02/08 11:45 True
2020/02/08 13:45 True
2020/02/08 17:00 False
2020/02/08 21:45 False
2020/02/09 00:45 False
2020/02/09 04:00 False
2020/02/09 07:45 False
2020/02/09 11:15 False
2020/02/09 12:15 False
2020/02/09 15:15 True
2020/02/10 02:00 True
2020/02/10 03:45 True
2020/02/10 04:15 True
2020/02